# Query Opteryx.app with SQLAlchemy

Connects to [opteryx.app](https://opteryx.app), the hosted service, with the `opteryx-sqlalchemy` dialect, queries the public **National Vulnerability Database (NVD)** dataset into pandas, then uses the same connection on data only your account can see.

This notebook accompanies [Using SQLAlchemy from a Notebook or Script](https://docs.opteryx.app/docs/guides/using-sqlalchemy) on docs.opteryx.app.

**You need an opteryx.app account.** The datasets under `public` are readable by any signed-in account, but SQLAlchemy goes through the Jobs API, and that API needs a credential for every query, public data included. Signing up is free at [opteryx.app](https://opteryx.app). Then create a personal access token in Opteryx Studio under **Settings → API Tokens**. It gives you a **client ID** and a **client secret**, and the secret is shown only once.

## 1. Install

In [ ]:
%pip install -q opteryx-sqlalchemy pandas matplotlib

## 2. Credentials

Keep the client secret out of the notebook itself, so it isn't saved with it or shared by accident. This cell looks for the client ID and secret in three places, in order:

1. **Colab Secrets.** Click the key icon in Colab's left sidebar, add `OPTERYX_CLIENT_ID` and `OPTERYX_CLIENT_SECRET`, and turn on notebook access for both. Secrets stay with your Colab account, so you set them once and every notebook can use them.
2. **Environment variables** with the same names, for running the notebook outside Colab.
3. **A prompt.** If neither is set, the cell asks, and the secret is typed into a hidden field.

In [ ]:
import os
from getpass import getpass


def credential(name, secret=False):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        pass   # not in Colab, or the secret isn't set or shared with this notebook
    if os.environ.get(name):
        return os.environ[name]
    return getpass(f"{name}: ") if secret else input(f"{name}: ")


CLIENT_ID = credential("OPTERYX_CLIENT_ID")
CLIENT_SECRET = credential("OPTERYX_CLIENT_SECRET", secret=True)
print("Client ID:", CLIENT_ID)

## 3. Connect

The client ID and secret go in the username and password slots of the connection URL. `URL.create` builds the URL and escapes any characters in the secret that would otherwise break it. The dialect exchanges the pair for a short-lived access token when the connection opens, so you never call the token endpoint yourself.

The first query counts the NVD records, which also checks that the credentials work. If it fails with `Authentication error`, opteryx.app didn't accept the client ID and secret. Check both values, and that the token hasn't been revoked.

In [ ]:
from sqlalchemy import URL, create_engine, text

engine = create_engine(
    URL.create(
        "opteryx",
        username=CLIENT_ID,
        password=CLIENT_SECRET,
        host="opteryx.app",
        port=443,
        database="default",
        query={"ssl": "true"},
    )
)

with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM public.security.nvd_vulnerabilities")).scalar()
print(f"{total:,} CVE records in public.security.nvd_vulnerabilities")

## 4. Query the public NVD dataset into pandas

`public.security.nvd_vulnerabilities` has one row per CVE: its ID, when it was published, its severity and CVSS score, and a description. `pandas.read_sql_query` runs a query and returns a DataFrame.

How many CVEs fall into each severity band:

In [ ]:
import pandas as pd

with engine.connect() as conn:
    by_severity = pd.read_sql_query(
        """
        SELECT severity, COUNT(*) AS cves
        FROM public.security.nvd_vulnerabilities
        GROUP BY severity
        ORDER BY cves DESC
        """,
        conn,
    )
by_severity

How many CVEs were published each year. The counting happens on opteryx.app, so only one row per year comes back to the notebook:

In [ ]:
with engine.connect() as conn:
    per_year = pd.read_sql_query(
        """
        SELECT EXTRACT(YEAR FROM published_at) AS year, COUNT(*) AS cves
        FROM public.security.nvd_vulnerabilities
        GROUP BY EXTRACT(YEAR FROM published_at)
        ORDER BY year
        """,
        conn,
    )
per_year.plot.bar(x="year", y="cves", legend=False, figsize=(12, 4), title="CVEs published per year")

The most recently published critical CVEs:

In [ ]:
with engine.connect() as conn:
    recent_critical = pd.read_sql_query(
        """
        SELECT cve_id, published_at, cvss_score, description
        FROM public.security.nvd_vulnerabilities
        WHERE severity = 'CRITICAL'
        ORDER BY published_at DESC
        LIMIT 10
        """,
        conn,
    )
recent_critical

Datasets can be joined like tables in any database. `public.security.cisa_kev` is CISA's catalog of vulnerabilities known to be exploited. Joining it to NVD on the CVE ID adds each one's CVSS score:

In [ ]:
with engine.connect() as conn:
    exploited = pd.read_sql_query(
        """
        SELECT kev.cve_id, kev.vendor_project, kev.product, kev.date_added, nvd.severity, nvd.cvss_score
        FROM public.security.cisa_kev AS kev
        INNER JOIN public.security.nvd_vulnerabilities AS nvd
            ON kev.cve_id = nvd.cve_id
        ORDER BY kev.date_added DESC
        LIMIT 10
        """,
        conn,
    )
exploited

## 5. The same connection on your own data

Nothing about the connection changes for data that isn't public. Each dataset has a dot-separated name, `workspace.collection.dataset`, and the workspace's policies decide what your credential can read. See [Access and Permissions](https://docs.opteryx.app/docs/core-concepts/access-and-permissions) for how.

This lists every schema (`workspace.collection`) your credential can see. It reads opteryx.app's metadata rather than running a query:

In [ ]:
from sqlalchemy import inspect

with engine.connect() as conn:
    schemas = inspect(conn).get_schema_names()
for schema in schemas:
    print(schema)

Pick one of your own datasets and query it exactly as you queried NVD. Set `MY_DATASET` to its full name, for example `acme.sales.orders`. If you haven't loaded anything yet, [Load and Query Data](https://docs.opteryx.app/docs/getting-started/reading-data) shows how.

In [ ]:
MY_DATASET = ""   # e.g. "acme.sales.orders"

if MY_DATASET:
    workspace_collection, dataset = MY_DATASET.rsplit(".", 1)
    with engine.connect() as conn:
        for column in inspect(conn).get_columns(dataset, schema=workspace_collection):
            print(f"{column['name']:<30} {column['type']}")
        mine = pd.read_sql_query(f"SELECT * FROM {MY_DATASET} LIMIT 10", conn)
    display(mine)
else:
    print("Set MY_DATASET to one of your own datasets, then run this cell again.")

## Things to know

- **Read-only.** `commit()` and `rollback()` do nothing, and the dialect doesn't support DDL or DML.
- **No bound parameters yet.** `text("... WHERE cvss_score > :min")` with a parameters dict fails on the server today. The [guide](https://docs.opteryx.app/docs/guides/using-sqlalchemy#parameterising-queries) shows the `SET` workaround, and when it is safe to use.
- **Large results.** `conn.execution_options(stream_results=True, max_row_buffer=500)` fetches a large result in pages, and `result_format="parquet"` fetches those pages as Parquet rather than JSON.
- **Rotating the secret.** If a secret leaks, revoke that token under **Settings → API Tokens** and create a new one.